# Agent Reasoning

**Module:** 10-agentic-ai-concepts

**Notebook:** `02-agent-reasoning.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Reasoning in Agents** with clear contracts and failure modes
- Explain and apply **Reactive Reasoning** with clear contracts and failure modes
- Explain and apply **Deliberative Reasoning** with clear contracts and failure modes
- Explain and apply **Internal Monologue Patterns** with clear contracts and failure modes
- Explain and apply **Failure Modes** with clear contracts and failure modes
- Explain and apply **Reasoning Eval Heuristics** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Agent Reasoning

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Reasoning in Agents**
2. **Reactive Reasoning**
3. **Deliberative Reasoning**
4. **Internal Monologue Patterns**
5. **Failure Modes**
6. **Reasoning Eval Heuristics**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Reasoning in Agents

### Definition
**Reasoning in Agents** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Reasoning in Agents typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Reasoning in Agents: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Reasoning in Agents as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Reasoning in Agents as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Reasoning in Agents
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Reasoning in Agents when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Reasoning in Agents improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Reasoning in Agents" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Reasoning in Agents"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Reactive Reasoning

### Definition
**Reactive Reasoning** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Reactive Reasoning typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Reactive Reasoning: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Reactive Reasoning as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Reactive Reasoning as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Reactive Reasoning
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Reactive Reasoning when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Reactive Reasoning" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Reactive Reasoning"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Reactive Reasoning

**Situation:** A team wants to productionize a feature involving **Reactive Reasoning**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Deliberative Reasoning

### Definition
**Deliberative Reasoning** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Deliberative Reasoning typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Deliberative Reasoning: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Deliberative Reasoning as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Deliberative Reasoning as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Deliberative Reasoning
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Deliberative Reasoning when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Deliberative Reasoning" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Deliberative Reasoning"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Deliberative Reasoning"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Deliberative Reasoning"}
strong = {"definition": "Deliberative Reasoning", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Deliberative Reasoning"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Deliberative Reasoning", "passed": len(checks)-len(failed), "failed": failed})


## Internal Monologue Patterns

### Definition
**Internal Monologue Patterns** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Internal Monologue Patterns typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Internal Monologue Patterns: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Internal Monologue Patterns as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Internal Monologue Patterns as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Internal Monologue Patterns
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Internal Monologue Patterns when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Internal Monologue Patterns" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Internal Monologue Patterns"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Internal Monologue Patterns"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Internal Monologue Patterns"}
strong = {"definition": "Internal Monologue Patterns", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Internal Monologue Patterns"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Internal Monologue Patterns", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Internal Monologue Patterns"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_internal_mon", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


### Worked scenario — Internal Monologue Patterns

**Situation:** A team wants to productionize a feature involving **Internal Monologue Patterns**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Failure Modes

### Definition
**Failure Modes** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Failure Modes typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Failure Modes: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Failure Modes as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Failure Modes as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Failure Modes
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Failure Modes when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Failure Modes" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Failure Modes"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Failure Modes"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Failure Modes"}
strong = {"definition": "Failure Modes", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Failure Modes"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Failure Modes", "passed": len(checks)-len(failed), "failed": failed})


## Reasoning Eval Heuristics

### Definition
**Reasoning Eval Heuristics** is a core building block in 02-agent-reasoning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Reasoning Eval Heuristics typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Reasoning Eval Heuristics: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Reasoning Eval Heuristics as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Reasoning Eval Heuristics as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Reasoning Eval Heuristics
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Reasoning Eval Heuristics when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Reasoning Eval Heuristics" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Reasoning Eval Heuristics"
    notebook: str = "02-agent-reasoning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Reasoning Eval Heuristics

**Situation:** A team wants to productionize a feature involving **Reasoning Eval Heuristics**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Agent Reasoning**.

| Topic | Do | Don't |
|-------|----|-------|
| Reasoning in Agents | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Reactive Reasoning | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Deliberative Reasoning | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Internal Monologue Patterns | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Failure Modes | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Reasoning Eval Heuristics | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Reasoning in Agents | Key concept covered in this notebook; see its section for definition and pitfalls |
| Reactive Reasoning | Key concept covered in this notebook; see its section for definition and pitfalls |
| Deliberative Reasoning | Key concept covered in this notebook; see its section for definition and pitfalls |
| Internal Monologue Patterns | Key concept covered in this notebook; see its section for definition and pitfalls |
| Failure Modes | Key concept covered in this notebook; see its section for definition and pitfalls |
| Reasoning Eval Heuristics | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Agent Reasoning** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **10-agentic-ai-concepts**.


## Try It Yourself

1. Implement a failing test/fixture for **Reasoning in Agents**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Reactive Reasoning**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Deliberative Reasoning**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Internal Monologue Patterns**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Failure Modes**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
